# 🕵️ Dia 16 — Modelagem de ameaças com STRIDE

**Investigador Cyber 90 Dias** · Fase 1 · Rank Jutsu `D`

> Baseado no *Day 16* de [cyber-investigator-skills](https://github.com/thiefdroppers/cyber-investigator-skills/blob/main/curriculum/phase-1-foundations/day-16.md), © ThiefDroppers (MIT).
> Notebook, camada BR e Camada 8 por Deivison Lourenço ([@Ridd1kulusC0d3r](https://github.com/Ridd1kulusC0d3r)).

---

### 🎬 Onde paramos

Quinta, 8/out. A Marta quer lançar amanhã um "Denuncie um golpe" no portal: telefone, link, print e CPF; a ouvidoria exporta uma planilha semanal por e-mail.

Rafa: *"E se o Pescador também gostar? Antes de lançar, a gente desenha. Cinco caixas, seis letras. Meia hora."*

Neste notebook você vai:

1. **Descrever os dois sistemas** (o banco de alimentos do original e o canal da Serra Azul) como elementos e fluxos.
2. **Aplicar a regra STRIDE por elemento** e gerar o esqueleto de ameaças.
3. **Desenhar os DFDs em Mermaid**, prontos para o seu repositório.
4. **Conferir o modelo contra o checkpoint** e jogar o quiz "qual letra?".

🐍 Só biblioteca padrão do Python. Nada para instalar. Rode as células de cima para baixo (Shift+Enter). Tudo funciona **sem internet**; quando uma célula tenta a rede, ela tem plano B embutido.

## 1. Os sistemas como dados

Um DFD é uma lista de elementos (com tipo e fronteira) e uma lista de fluxos (de, para, rótulo). Descrito assim, ele vira algo que o código consegue percorrer.

In [ ]:
# Tipos: "entidade", "processo", "armazenamento", "log" (armazenamento que é log)
# Cada elemento: nome -> (tipo, fronteira)   (fronteira None = fora de qualquer fronteira)

FOODBANK = {
    "elementos": {
        "Volunteer (browser)":           ("entidade", None),
        "Coordinator (browser)":         ("entidade", None),
        "Email service (third party)":   ("entidade", "Serviço de e-mail de terceiro"),
        "Web app (nginx + Python)":      ("processo", "Servidor do banco de alimentos"),
        "Sign-up database (PostgreSQL)": ("armazenamento", "Servidor do banco de alimentos"),
        "nginx access log":              ("log", "Servidor do banco de alimentos"),
        "App config file (API key)":     ("armazenamento", "Servidor do banco de alimentos"),
    },
    "fluxos": [
        ("Volunteer (browser)", "Web app (nginx + Python)", "Formulário de inscrição (nome, e-mail, telefone, turnos), HTTPS"),
        ("Web app (nginx + Python)", "Sign-up database (PostgreSQL)", "INSERT da inscrição"),
        ("Web app (nginx + Python)", "Email service (third party)", "Pedido de e-mail de confirmação + chave de API, HTTPS"),
        ("Email service (third party)", "Volunteer (browser)", "E-mail de confirmação"),
        ("Coordinator (browser)", "Web app (nginx + Python)", "Login de administração (usuário, senha)"),
        ("Web app (nginx + Python)", "Coordinator (browser)", "Lista de inscrições / exportação CSV"),
        ("Web app (nginx + Python)", "nginx access log", "Registros de requisição"),
        ("App config file (API key)", "Web app (nginx + Python)", "Chave de API, credenciais do banco"),
    ],
}

DENUNCIE = {
    "elementos": {
        "Cooperado (celular)":            ("entidade", None),
        "Atendente ouvidoria (navegador)": ("entidade", "Rede interna da cooperativa"),
        "Gateway de SMS (terceiro)":      ("entidade", "Gateway de SMS de terceiro"),
        "Vigia (e-mail)":                 ("entidade", "Vigia"),
        "Portal / denuncie (nginx + app)": ("processo", "sa-web01"),
        "Banco de denúncias":             ("armazenamento", "sa-web01"),
        "Pasta de uploads (prints)":      ("armazenamento", "sa-web01"),
        "Config (chave do gateway)":      ("armazenamento", "sa-web01"),
        "nginx access log":               ("log", "sa-web01"),
    },
    "fluxos": [
        ("Cooperado (celular)", "Portal / denuncie (nginx + app)", "Denúncia: telefone, link, texto, print, CPF, HTTPS"),
        ("Portal / denuncie (nginx + app)", "Banco de denúncias", "INSERT da denúncia"),
        ("Portal / denuncie (nginx + app)", "Pasta de uploads (prints)", "Arquivo do print"),
        ("Portal / denuncie (nginx + app)", "Gateway de SMS (terceiro)", "Pedido de SMS de protocolo + chave de API"),
        ("Gateway de SMS (terceiro)", "Cooperado (celular)", "SMS 'Recebemos sua denúncia, protocolo N'"),
        ("Atendente ouvidoria (navegador)", "Portal / denuncie (nginx + app)", "Login da intranet (usuário, senha)"),
        ("Portal / denuncie (nginx + app)", "Atendente ouvidoria (navegador)", "Lista de denúncias, prints e planilha semanal"),
        ("Atendente ouvidoria (navegador)", "Vigia (e-mail)", "Planilha semanal por e-mail"),
        ("Portal / denuncie (nginx + app)", "nginx access log", "Registros de requisição"),
        ("Config (chave do gateway)", "Portal / denuncie (nginx + app)", "Chave do gateway, credenciais do banco"),
    ],
}
print("Banco de alimentos:", len(FOODBANK["elementos"]), "elementos,", len(FOODBANK["fluxos"]), "fluxos")
print("Denuncie um golpe:  ", len(DENUNCIE["elementos"]), "elementos,", len(DENUNCIE["fluxos"]), "fluxos")

## 2. STRIDE por elemento: o esqueleto

A regra do README em forma de dicionário. Fluxos que **cruzam fronteira** (as duas pontas em fronteiras diferentes) aparecem marcados: é onde mora a maioria das ameaças.

In [ ]:
REGRA = {"entidade": "SR", "processo": "STRIDE", "armazenamento": "TID", "log": "TIDR", "fluxo": "TID"}
NOMES = {"S": "Spoofing", "T": "Tampering", "R": "Repudiation", "I": "Information disclosure",
         "D": "Denial of service", "E": "Elevation of privilege"}

def esqueleto(sistema):
    """Devolve a lista de (letra, alvo, cruza_fronteira) que o modelo precisa considerar."""
    perguntas = []
    for nome, (tipo, _) in sistema["elementos"].items():
        for letra in REGRA[tipo]:
            perguntas.append((letra, nome, False))
    for de, para, rotulo in sistema["fluxos"]:
        f_de = sistema["elementos"][de][1]
        f_para = sistema["elementos"][para][1]
        cruza = f_de != f_para                       # pontas em fronteiras diferentes
        for letra in REGRA["fluxo"]:
            perguntas.append((letra, f"{de} → {para} ({rotulo})", cruza))
    return perguntas

esq = esqueleto(DENUNCIE)
print(f"{len(esq)} perguntas para o canal 'Denuncie um golpe'. Algumas, das que cruzam fronteira:\n")
for letra, alvo, cruza in esq:
    if cruza and letra == "I":
        print(f"  [{letra}] {alvo}")

> 💡 Isto é um **esqueleto**, não um modelo. Cada pergunta só vira ameaça quando alguém escreve uma descrição que nomeia o campo, a página ou o componente e propõe uma mitigação. O código faz a lista; você faz o pensamento.

## 3. Os DFDs em Mermaid

In [ ]:
import re

def mermaid(sistema, titulo):
    """Gera o DFD com as formas usuais: [ ] entidade, (( )) processo, [( )] armazenamento."""
    ids = {nome: f"N{i}" for i, nome in enumerate(sistema["elementos"])}
    forma = {"entidade": '["{}"]', "processo": '(("{}"))', "armazenamento": '[("{}")]', "log": '[("{}")]'}
    linhas = ["flowchart LR", f"    %% {titulo}"]
    fronteiras = {}
    for nome, (tipo, fr) in sistema["elementos"].items():
        no = ids[nome] + forma[tipo].format(nome)
        fronteiras.setdefault(fr, []).append(no)
    for i, (fr, nos) in enumerate(fronteiras.items()):
        if fr is None:
            linhas += [f"    {n}" for n in nos]
        else:
            linhas.append(f'    subgraph F{i}["Fronteira de confiança: {fr}"]')
            linhas += [f"        {n}" for n in nos]
            linhas.append("    end")
            linhas.append(f"    style F{i} stroke-dasharray: 6 4")
    for de, para, rotulo in sistema["fluxos"]:
        linhas.append(f'    {ids[de]} -->|"{rotulo}"| {ids[para]}')
    return "\n".join(linhas)

DFD_DENUNCIE = mermaid(DENUNCIE, "Serra Azul — Denuncie um golpe")
print("```mermaid")
print(DFD_DENUNCIE)
print("```")

## 4. As ameaças do canal da Serra Azul

Já vêm preenchidas as três obrigatórias do README e mais cinco de exemplo. **Reescreva pelo menos duas com as suas palavras** e, se quiser, acrescente outras: a célula seguinte confere o modelo contra o checkpoint.

In [ ]:
#@title Ameaças do canal "Denuncie um golpe" { display-mode: "form" }
fluxo_maior_risco = "Atendente ouvidoria → Vigia (planilha semanal por e-mail): sai de dentro da cooperativa para fora, com telefone, texto e CPF de todos os denunciantes da semana."  #@param {type:"string"}

# (letra, alvo, título, descrição específica, severidade, mitigação)
AMEACAS = [
    ("S", "Cooperado (celular)", "'Denuncie' falso por SMS",
     "O Pescador registra serraazul-denuncie.example e manda SMS 'Denuncie o golpe aqui'; o cooperado não distingue o canal verdadeiro.",
     "High", "Divulgar o canal só no app e no portal; aviso público de que a Serra Azul nunca manda link de denúncia por SMS; monitorar o domínio (Dia 13)."),
    ("I", "Atendente ouvidoria → Vigia (planilha)", "Planilha semanal com CPF por e-mail",
     "A planilha leva telefone, texto e CPF de todos os denunciantes; uma caixa de e-mail comprometida expõe todos.",
     "High", "Não coletar o CPF (o protocolo resolve o retorno); enviar só os campos que a Vigia usa (telefone remetente, link, texto, hora); canal com acesso controlado em vez de e-mail."),
    ("R", "Portal / denuncie (nginx + app)", "Sem log de auditoria de denúncias e exportações",
     "Ninguém prova quando uma denúncia chegou, de qual sessão, nem qual atendente exportou a planilha; o nginx só mostra POST /denuncie e um IP.",
     "Medium", "Log de auditoria: evento, protocolo, conta do atendente, IP de origem, hora UTC; enviado para fora de sa-web01."),
    ("T", "Pasta de uploads (prints)", "Print que não é imagem",
     "Um 'print' enviado pelo formulário é um arquivo malicioso (ou imagem malformada) que o atendente abre na estação da agência.",
     "High", "Aceitar só PNG/JPEG conferindo o conteúdo (não a extensão), reprocessar a imagem no servidor, abrir prints num visualizador isolado."),
    ("D", "Portal / denuncie (nginx + app)", "Enxurrada de denúncias falsas",
     "Milhares de envios automatizados enchem o banco e a pasta de uploads e escondem as denúncias reais.",
     "Medium", "Limite de taxa por IP e por telefone, tamanho máximo de arquivo, CAPTCHA acessível."),
    ("E", "Portal / denuncie (nginx + app)", "Área da ouvidoria protegida só por link",
     "Se a lista de denúncias depender de uma URL não divulgada e não do papel da sessão, qualquer um que a descubra lê tudo.",
     "High", "Conferir papel 'ouvidoria' em toda rota restrita; segundo fator para atendentes."),
    ("I", "Config (chave do gateway)", "Chave do gateway de SMS vazada",
     "Com a chave, alguém manda SMS que saem pelo remetente da cooperativa: o golpe perfeito.",
     "High", "Chave fora do código, com permissão mínima; restrição de IP de origem no gateway; rotação; alerta de volume."),
    ("S", "Atendente ouvidoria (navegador)", "Login de atendente por credential stuffing",
     "Usuário e senha da intranet, sem segundo fator, expostos à internet pelo portal.",
     "High", "Segundo fator; área restrita só pela rede interna ou VPN; registro de falhas."),
]

ok_letras = set(a[0] for a in AMEACAS)
problemas = []
if len(AMEACAS) < 8:
    problemas.append(f"só {len(AMEACAS)} ameaças (mínimo 8)")
if ok_letras != set("STRIDE"):
    problemas.append("faltam letras: " + ", ".join(sorted(set("STRIDE") - ok_letras)))
for a in AMEACAS:
    if not a[5].strip():
        problemas.append(f"'{a[2]}' sem mitigação")
if not any(a[0] == "R" and "log" in a[5].lower() for a in AMEACAS):
    problemas.append("nenhuma ameaça de repúdio aponta o log que falta")
if not fluxo_maior_risco.strip():
    problemas.append("falta o fluxo de maior risco")

print("✅ Modelo passa nos critérios do checkpoint." if not problemas else "❌ Ajustar:")
for p in problemas:
    print("  -", p)
print("\nCobertura STRIDE:", " ".join(f"{l}✔" if l in ok_letras else f"{l}✘" for l in "STRIDE"))

## 5. 🎮 Quiz: qual letra?

In [ ]:
MODO_JOGO = True

QUIZ = [
    ("Alguém entra como coordenador com senha vazada de outro site", "S"),
    ("<script> no campo nome, exibido depois na página de administração", "T"),
    ("O coordenador nega ter exportado a lista, e nada prova o contrário", "R"),
    ("signups-2026-03-10.csv pode ser baixado por quem adivinha a data", "I"),
    ("Milhares de inscrições automatizadas derrubam o formulário", "D"),
    ("A página de admin só depende de um link escondido", "E"),
    ("SMS 'Denuncie o golpe aqui' com link de domínio imitação", "S"),
    ("Planilha da ouvidoria com CPF vai por e-mail", "I"),
    ("Um 'print' que é um arquivo malicioso, aberto na agência", "T"),
    ("Não há como provar quando uma denúncia chegou", "R"),
]
acertos = 0
for i, (ameaca, certa) in enumerate(QUIZ, start=1):
    print(f"\n[{i:02d}/{len(QUIZ)}] {ameaca}")
    if MODO_JOGO:
        r = ""
        while r not in list("STRIDE"):
            r = input("   Letra STRIDE (S/T/R/I/D/E): ").strip().upper()[:1]
            if r == "":
                r = "?"
        acertos += (r == certa)
        print("   ✅" if r == certa else f"   ❌ Era {certa}.")
    print(f"   {certa} = {NOMES[certa]}")
if MODO_JOGO:
    print(f"\n🏁 {acertos}/{len(QUIZ)}", "— Checkpoint BR cumprido." if acertos >= 8 else "— Meta: 8.")

## 6. 📝 Salvar o modelo em Markdown (com o DFD)

In [ ]:
from pathlib import Path
md = ["# Modelo de ameaças — Serra Azul, canal 'Denuncie um golpe' (Dia 16)", "",
      "## DFD", "", "```mermaid", DFD_DENUNCIE, "```", "",
      "## Fluxo de maior risco", fluxo_maior_risco, "",
      "## Ameaças", "", "| # | STRIDE | Alvo | Título | Descrição | Severidade | Mitigação | Status |",
      "|---|---|---|---|---|---|---|---|"]
for i, (l, alvo, t, d, sev, mit) in enumerate(AMEACAS, start=1):
    md.append(f"| A{i:02d} | {l} | {alvo} | {t} | {d} | {sev} | {mit} | Open |")
Path("notas").mkdir(exist_ok=True)
Path("notas/isca-dia16-denuncie.md").write_text("\n".join(md) + "\n", encoding="utf-8")
print("\n".join(md[:12]))
print("...\n💾 notas/isca-dia16-denuncie.md (as IDs A01–A08 são usadas no Dia 17)")
try:
    from google.colab import files
    files.download("notas/isca-dia16-denuncie.md")
except ImportError:
    print("(Fora do Colab: o arquivo ficou em notas/)")

## ✅ Antes de fechar o Dia 16

- [ ] Quiz com pelo menos 8/10
- [ ] `notas/isca-dia16-denuncie.md` com 8+ ameaças e as 6 letras
- [ ] Os dois modelos no Threat Dragon, com JSON, PDF e PNG (README, Passos 1 a 6)
- [ ] `status` do dia 16 = `estudado` no `curriculo/mapa-dos-dias.csv`

**Amanhã (Dia 17):** do modelo de ameaças ao mapa de evidências. Para cada ameaça de hoje: qual registro provaria que ela aconteceu, e ele existe?

---
<sub>Baseado em cyber-investigator-skills por ThiefDroppers (MIT) · Investigador Cyber 90 Dias por Deivison Lourenço</sub>